In [2]:
# =============================================================================
# CELL 1: IMPORTS AND CONFIGURATION
# =============================================================================

import pandas as pd
import numpy as np
import pickle
import os
import openpyxl

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 500)
pd.set_option('display.float_format', '{:.4f}'.format)

def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

print('Imports ready')

Imports ready


In [3]:
# =============================================================================
# CELL 2: LOAD DLA DATA FROM CACHE
# =============================================================================

dla_df = get_pickle('../../cache/dla_v1.pkl')

print(f'DLA records: {len(dla_df):,}')
print(f'Columns: {list(dla_df.columns)}')
print(f'Unique vintages: {sorted(dla_df.valid_vintage.unique())}')
print(f'Unique dealer count: {dla_df.dealer_number.nunique():,}')
display(dla_df.head(10))

DLA records: 134,149
Columns: ['dll_edition', 'dealer_number', 'valid_vintage', 'loss_ratio', 'dealer_level', 'pricing_scalar']
Unique vintages: ['2019 Q1', '2019 Q2', '2019 Q3', '2019 Q4', '2020 Q1', '2020 Q2', '2020 Q3', '2020 Q4', '2021 Q1', '2021 Q2', '2021 Q3', '2021 Q4', '2022 Q1', '2022 Q2', '2022 Q3', '2022 Q4', '2023 Q1', '2023 Q2', '2023 Q3', '2023 Q4', '2024 Q1', '2024 Q2', '2024 Q3', '2024 Q4', '2025 Q1', '2025 Q2', '2025 Q3', '2025 Q4', 'current']
Unique dealer count: 7,305


,dll_edition,dealer_number,valid_vintage,loss_ratio,dealer_level,pricing_scalar
0,2025 Q4 FRN4.0,1051,2019 Q1,1.0000,C,1.0000
1,2025 Q4 FRN4.0,1051,2019 Q2,1.0000,C,1.0000
2,2025 Q4 FRN4.0,1051,2019 Q3,1.0000,C,1.0000
3,2025 Q4 FRN4.0,1051,2019 Q4,1.0000,C,1.0000
4,2025 Q4 FRN4.0,1051,2020 Q1,1.0000,C,1.0000
5,2025 Q4 FRN4.0,1051,2020 Q2,1.0000,C,1.0000
6,2025 Q4 FRN4.0,1051,2024 Q2,1.0000,C,1.0000
7,2025 Q4 FRN4.0,1051,2024 Q3,1.0000,C,1.0000
8,2025 Q4 FRN4.0,1100,2022 Q2,1.0000,C,1.0000
9,2025 Q4 FRN4.0,1100,2022 Q3,1.0000,C,1.0000


In [4]:
# =============================================================================
# CELL 3: USE CURRENT DLA SCALARS
# =============================================================================

# Use the 'current' vintage for the most up-to-date pricing scalar.
# Fall back to the latest explicit vintage if 'current' is unavailable.
if 'current' in dla_df.valid_vintage.values:
    dla_current = dla_df[dla_df.valid_vintage == 'current'].copy()
    print(f'Using "current" vintage: {len(dla_current):,} dealers')
else:
    latest = dla_df.valid_vintage.max()
    dla_current = dla_df[dla_df.valid_vintage == latest].copy()
    print(f'Using latest vintage "{latest}": {len(dla_current):,} dealers')

dla_current = dla_current.drop_duplicates(subset='dealer_number', keep='last')
print(f'Unique dealers with scalars: {len(dla_current):,}')
print(f'Scalar range: {dla_current.pricing_scalar.min():.4f} to {dla_current.pricing_scalar.max():.4f}')
display(dla_current.describe())

Using "current" vintage: 7,305 dealers
Unique dealers with scalars: 7,305
Scalar range: 0.6500 to 1.5000


,dealer_number,loss_ratio,pricing_scalar
count,7305.0000,7305.0000,7305.0000
mean,29519.2812,0.9573,0.9855
std,11938.4138,0.1734,0.1121
min,52.0000,0.0000,0.6500
25%,25753.0000,0.9383,1.0000
50%,30041.0000,1.0000,1.0000
75%,32839.0000,1.0000,1.0000
max,62846.0000,1.9442,1.5000


In [5]:
# =============================================================================
# CELL 4: DEFINE CORPORATE DEALER GROUPS WITH DEALER NUMBER RANGES
# =============================================================================

def expand_dealer_spec(spec_str):
    """Parse a dealer specification string into a list of dealer numbers.
    Commas separate entries. Dashes/en-dashes indicate inclusive ranges.
    """
    dealers = []
    parts = [p.strip() for p in spec_str.split(',')]
    for part in parts:
        part = part.replace('\u2013', '-').replace('\u2014', '-')  # en-dash, em-dash
        if '-' in part:
            tokens = part.split('-')
            start, end = int(tokens[0].strip()), int(tokens[-1].strip())
            dealers.extend(range(start, end + 1))
        else:
            dealers.append(int(part))
    return dealers


DEALER_GROUPS = {
    'Auto Boutique': '29951, 30405, 31477',
    'Avis': '28558, 28563, 28812-28822, 28827-28829, 29005-29011, 29533, 30282, 33347',
    'EchoPark': '29196-29198, 31182, 31187, 31193, 31196, 31205, 31708, 31776, 33106',
    'HGreg': '28347-28351, 29646-29647, 29829, 29842-29843, 30211, 33325',
    'Hertz Car Sales': '27412-27419, 27424-27426, 28234, 28411-28536, 28883, 29013-29022, 29452, 29609, 30663-30669, 31169, 32845, 58934-58935, 60103, 60107',
    'Penske': '6333, 16099, 16102, 16103, 23328, 23329, 23330, 23331, 23332, 23334, 23335, 23336, 23337, 23339, 23340, 23341, 23342, 23343, 23383, 23397, 23398, 23399, 23400, 23401, 23402, 23403, 23418, 23430, 23431, 23432, 23433, 23434, 23435, 23436, 23459, 23464, 23465, 23466, 23467, 23468, 23469, 23471, 23472, 23473, 23500, 23523, 23669, 23670, 23671, 23672, 23674, 23675, 23677, 23707, 23708, 23709, 23710, 23711, 23712, 23713, 23726, 23728, 23729, 23730, 23731, 23732, 23733, 23734, 23735, 23736, 23737, 23738, 23757, 23758, 23759, 23760, 23762, 23763, 23765, 23766, 23767, 23768, 23769, 23770, 23771, 23772, 23773, 23774, 23775, 23776, 23777, 23815, 23816, 23817, 23818, 23819, 23820, 23821, 23822, 23823, 23824, 23825, 23826, 23827, 23828, 23829, 23830, 23831, 23832, 23833, 23834, 23835, 24323, 24562, 24563, 24564, 25203, 26075, 26204, 26205, 26206, 26207, 26208, 26291, 26346, 26639, 26781, 27148, 27483, 27484, 28713, 29786, 29955, 30198, 30281, 30639, 30944, 30945, 31088, 31089, 31287, 31288, 31316, 31742, 32396, 32893, 32894, 33386, 33431, 33432, 39055, 39273, 45038, 53024, 55250, 55482, 63098',
    'Woodhouse Auto Family': '28919, 28959-28967, 29282-29284, 29937, 31007, 32131, 32466, 32866, 40301, 45656, 46274-46275, 47115, 53065-53066',
}

SONIC_DEALERS = [
    18, 61, 66, 91, 96, 102, 136, 226, 262, 341, 400, 401, 405, 406, 408,
    411, 412, 413, 414, 415, 416, 426, 432, 433, 435, 436, 437, 442, 443,
    444, 445, 446, 447, 450, 454, 455, 456, 457, 458, 459, 460, 467, 468,
    469, 475, 478, 480, 482, 483, 484, 485, 486, 487, 489, 490, 491, 492,
    493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506,
    507, 508, 509, 510, 511, 512, 513, 515, 516, 517, 518, 523, 524, 527,
    528, 529, 530, 531, 532, 533, 535, 537, 538, 539, 541, 542, 543, 560,
    635, 636, 638, 639, 640, 719, 734, 758, 789, 799, 938, 942, 1080, 1091,
    1121, 8395, 29127, 29128, 29129, 29130, 29131, 29132, 29133, 29134,
    29135, 29136, 29137, 29139, 29140, 29141, 29142, 29143, 29144, 29145,
    29147, 29148, 29149, 29150, 29151, 29152, 29153, 29154, 29155, 29156,
    29157, 29158, 29159, 29160, 29161, 29162, 29165, 29166, 29167, 29168,
    29169, 29170, 29171, 29172, 29173, 29174, 29175, 29176, 29177, 29178,
    29179, 29180, 29181, 29182, 29183, 29184, 29185, 29186, 29187, 29188,
    29189, 29190, 29191, 29192, 29193, 29194, 29195, 29196, 29197, 29198,
    29199, 29200, 29202, 29203, 29204, 29205, 29206, 29207, 29208, 29209,
    29210, 29211, 29212, 29213, 29215, 29216, 29217, 29218, 29483, 30266,
    30469, 30582, 30593, 30600, 30746, 30961, 30962, 30963, 30964, 30965,
    30966, 30967, 30968, 30969, 30970, 30971, 30972, 30973, 30974, 30975,
    30976, 30977, 30978, 30979, 30980, 30981, 30982, 30983, 30984, 30985,
    31011, 31182, 31183, 31184, 31185, 31186, 31187, 31188, 31189, 31190,
    31191, 31192, 31193, 31194, 31195, 31196, 31197, 31198, 31199, 31200,
    31201, 31202, 31203, 31204, 31205, 31206, 31207, 31208, 31209, 31210,
    31211, 31212, 31213, 31214, 31215, 31350, 31776, 32382, 33106, 35549,
    35550, 35551, 35552, 35553, 35554, 35555, 35556, 35557, 37913, 41945,
    49860, 49927, 59479, 59746, 59751, 59752, 59753, 67049, 67051, 67053,
    67054, 67057,
]

expanded_groups = {}
for name, spec in DEALER_GROUPS.items():
    expanded = expand_dealer_spec(spec)
    expanded_groups[name] = expanded
    print(f'{name}: {len(expanded):,} dealer numbers in spec')

expanded_groups['Sonic Automotive'] = SONIC_DEALERS
print(f'Sonic Automotive: {len(SONIC_DEALERS):,} dealer numbers (explicit list)')

print(f'\nTotal groups: {len(expanded_groups)}')

Auto Boutique: 3 dealer numbers in spec
Avis: 26 dealer numbers in spec
EchoPark: 11 dealer numbers in spec
HGreg: 12 dealer numbers in spec
Hertz Car Sales: 164 dealer numbers in spec
Penske: 157 dealer numbers in spec
Woodhouse Auto Family: 25 dealer numbers in spec
Sonic Automotive: 295 dealer numbers (explicit list)

Total groups: 8


In [6]:
# =============================================================================
# CELL 5: MATCH DEALERS WITH DLA DATA AND COMPUTE SCALARS
# =============================================================================

all_rows = []

for group_name, dealer_list in expanded_groups.items():
    matched = dla_current[dla_current.dealer_number.isin(dealer_list)].copy()
    matched['corporate_group'] = group_name

    # Dealers in the spec range that have no DLA scalar
    missing = set(dealer_list) - set(matched.dealer_number.tolist())

    print(f'{group_name}: {len(matched)} dealers matched in DLA '
          f'(out of {len(dealer_list):,} in spec, {len(missing):,} not found in DLA)')

    all_rows.append(matched)

dealer_results = pd.concat(all_rows, ignore_index=True)
print(f'\nTotal matched dealers: {len(dealer_results):,}')

Auto Boutique: 3 dealers matched in DLA (out of 3 in spec, 0 not found in DLA)
Avis: 20 dealers matched in DLA (out of 26 in spec, 6 not found in DLA)
EchoPark: 8 dealers matched in DLA (out of 11 in spec, 3 not found in DLA)
HGreg: 10 dealers matched in DLA (out of 12 in spec, 2 not found in DLA)
Hertz Car Sales: 111 dealers matched in DLA (out of 164 in spec, 53 not found in DLA)
Penske: 149 dealers matched in DLA (out of 157 in spec, 8 not found in DLA)
Woodhouse Auto Family: 24 dealers matched in DLA (out of 25 in spec, 1 not found in DLA)
Sonic Automotive: 147 dealers matched in DLA (out of 295 in spec, 148 not found in DLA)

Total matched dealers: 472


In [7]:
# =============================================================================
# CELL 6: PER-DEALER OUTPUT TABLE
# =============================================================================

output_df = dealer_results[['dealer_number', 'corporate_group', 'pricing_scalar',
                            'dealer_level', 'loss_ratio']].copy()
output_df = output_df.rename(columns={
    'dealer_number': 'Dealer Number',
    'corporate_group': 'Corporate Dealer Group',
    'pricing_scalar': 'DLA Loss Scalar',
    'dealer_level': 'Dealer Level',
    'loss_ratio': 'Loss Ratio',
})
output_df = output_df.sort_values(['Corporate Dealer Group', 'Dealer Number']).reset_index(drop=True)

print(f'Output table: {len(output_df)} rows')
display(output_df)

Output table: 472 rows


,Dealer Number,Corporate Dealer Group,DLA Loss Scalar,Dealer Level,Loss Ratio
0,29951,Auto Boutique,0.7000,A,0.4828
1,30405,Auto Boutique,0.7000,A,0.6476
2,31477,Auto Boutique,1.0000,C,0.8580
3,28558,Avis,1.0000,C,1.0952
4,28563,Avis,1.0000,C,1.0952
5,28812,Avis,1.0000,C,1.0952
6,28813,Avis,1.2500,D,0.9594
7,28814,Avis,1.0000,C,1.0952
8,28815,Avis,1.0000,C,1.0952
9,28816,Avis,1.0000,C,1.0952


In [8]:
# =============================================================================
# CELL 7: AVERAGE DLA LOSS SCALAR PER CORPORATE DEALER GROUP BY QUARTER
# =============================================================================

# Build a mapping from dealer_number -> corporate group
dealer_to_group = {}
for group_name, dealer_list in expanded_groups.items():
    for d in dealer_list:
        dealer_to_group[d] = group_name

# Use the full DLA dataset (all vintages, not just 'current')
dla_explicit = dla_df[dla_df.valid_vintage != 'current'].copy()
dla_explicit = dla_explicit[dla_explicit.dealer_number.isin(dealer_to_group)]
dla_explicit['corporate_group'] = dla_explicit.dealer_number.map(dealer_to_group)

# Filter to quarters >= 2024 Q1
dla_explicit = dla_explicit[dla_explicit.valid_vintage >= '2024 Q1'].copy()

# Also add 'current' as its own column
dla_curr = dla_df[dla_df.valid_vintage == 'current'].copy()
dla_curr = dla_curr[dla_curr.dealer_number.isin(dealer_to_group)]
dla_curr['corporate_group'] = dla_curr.dealer_number.map(dealer_to_group)

# Average scalar per group per vintage
vintage_avg = dla_explicit.groupby(['corporate_group', 'valid_vintage'])['pricing_scalar'].mean()
vintage_avg = vintage_avg.unstack('valid_vintage')

# Average scalar for 'current'
current_avg = dla_curr.groupby('corporate_group')['pricing_scalar'].mean()
vintage_avg['current'] = current_avg

# Dealer count per group (from 'current' vintage)
dealer_counts = dla_curr.groupby('corporate_group')['dealer_number'].nunique()
vintage_avg.insert(0, 'Dealers Matched', dealer_counts)

vintage_avg = vintage_avg.rename_axis('Corporate Dealer Group')
vintage_avg = vintage_avg.sort_index()

print('=== Average DLA Loss Scalar per Corporate Dealer Group by Quarter ===')
display(vintage_avg)

=== Average DLA Loss Scalar per Corporate Dealer Group by Quarter ===


valid_vintage,Dealers Matched,2024 Q1,2024 Q2,2024 Q3,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4,current
Corporate Dealer Group,,,,,,,,,,
Auto Boutique,3,0.8500,0.8500,0.8500,1.0333,0.9833,0.8500,0.8000,0.8000,0.8000
Avis,20,1.0325,0.9525,0.9325,1.0375,1.0450,1.4000,0.8700,1.0000,1.0125
EchoPark,1,0.8500,0.8500,0.8500,0.8500,0.8500,0.8500,1.0000,1.0000,1.0000
HGreg,10,1.0500,0.9950,1.0100,0.9950,0.9800,1.0400,1.0300,0.9850,1.0100
Hertz Car Sales,111,1.0088,1.0055,0.8940,0.8894,0.8757,0.8725,0.8730,0.8820,0.9131
Penske,149,0.8708,0.8837,0.9895,0.8622,0.8652,0.8732,0.8678,1.0309,1.0279
Sonic Automotive,147,1.0132,0.9985,0.9824,0.9654,0.9602,0.9549,0.9455,0.9660,0.9677
Woodhouse Auto Family,24,0.9889,0.9750,0.8737,1.0262,0.9478,0.8500,0.7500,0.7500,0.7375


In [9]:
# =============================================================================
# CELL 7.5: MAP AVERAGE SCALAR TO CLOSEST DEALER BUCKET (A-E)
# =============================================================================

# Extract the scalar-to-bucket mapping from the DLA data itself
bucket_map = (
    dla_df[dla_df.valid_vintage == 'current']
    .groupby('dealer_level')['pricing_scalar']
    .median()
    .sort_values()
)
print('Dealer Level -> Median Scalar (from DLA data):')
for level, scalar in bucket_map.items():
    print(f'  {level} = {scalar:.4f}')

bucket_levels = bucket_map.index.tolist()
bucket_scalars = bucket_map.values

def scalar_to_bucket(val):
    if pd.isna(val):
        return np.nan
    idx = np.argmin(np.abs(bucket_scalars - val))
    return bucket_levels[idx]

# Apply to the vintage_avg table (skip 'Dealers Matched' column)
scalar_cols = [c for c in vintage_avg.columns if c != 'Dealers Matched']
bucket_df = vintage_avg[scalar_cols].map(scalar_to_bucket)
bucket_df.insert(0, 'Dealers Matched', vintage_avg['Dealers Matched'])

print('\n=== Average DLA Scalar Mapped to Nearest Dealer Bucket ===')
display(bucket_df)

Dealer Level -> Median Scalar (from DLA data):
  A = 0.7000
  B = 0.8500
  C = 1.0000
  D = 1.2000
  E = 1.3500
  F = 1.5000

=== Average DLA Scalar Mapped to Nearest Dealer Bucket ===


valid_vintage,Dealers Matched,2024 Q1,2024 Q2,2024 Q3,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4,current
Corporate Dealer Group,,,,,,,,,,
Auto Boutique,3,B,B,B,C,C,B,B,B,B
Avis,20,C,C,C,C,C,E,B,C,C
EchoPark,1,B,B,B,B,B,B,C,C,C
HGreg,10,C,C,C,C,C,C,C,C,C
Hertz Car Sales,111,C,C,B,B,B,B,B,B,B
Penske,149,B,B,C,B,B,B,B,C,C
Sonic Automotive,147,C,C,C,C,C,C,C,C,C
Woodhouse Auto Family,24,C,C,B,C,C,B,A,A,A


In [10]:
# =============================================================================
# CELL 7.6: WEIGHTED AVERAGE DLA SCALAR (WEIGHTED BY AMOUNT FINANCED)
# =============================================================================

# Load ULA data to get amt_financed per loan
ula_raw = get_pickle('../../cache/ula_v1.pkl')
ula_raw = ula_raw[ula_raw.lob != 'Core']

# Keep only columns needed: dealer_number, book_vintage (quarter), amt_financed
ula_slim = ula_raw[['dealer_number', 'book_date', 'amt_financed']].copy()
ula_slim['book_date'] = pd.to_datetime(ula_slim['book_date'])
ula_slim['book_vintage'] = ula_slim['book_date'].dt.to_period('Q').astype(str)
ula_slim = ula_slim[ula_slim.book_vintage >= '2024Q1']

# Filter to dealers in our groups
ula_slim = ula_slim[ula_slim.dealer_number.isin(dealer_to_group)]
ula_slim['corporate_group'] = ula_slim.dealer_number.map(dealer_to_group)

# Merge DLA scalars onto ULA loans by dealer_number + book_vintage
dla_renamed = dla_df.rename(columns={'valid_vintage': 'book_vintage'})
dla_expl = dla_renamed[dla_renamed.book_vintage != 'current'][['dealer_number', 'book_vintage', 'pricing_scalar']]
dla_curr_fallback = dla_renamed[dla_renamed.book_vintage == 'current'][['dealer_number', 'pricing_scalar']]
dla_curr_fallback = dla_curr_fallback.rename(columns={'pricing_scalar': 'pricing_scalar_current'})

# Standardize vintage format: ULA uses '2024Q1', DLA uses '2024 Q1'
ula_slim['book_vintage'] = ula_slim['book_vintage'].str.replace(r'(\d{4})Q(\d)', r'\1 Q\2', regex=True)

ula_merged = ula_slim.merge(dla_expl, on=['dealer_number', 'book_vintage'], how='left')
ula_merged = ula_merged.merge(dla_curr_fallback, on='dealer_number', how='left')
ula_merged['pricing_scalar'] = ula_merged['pricing_scalar'].fillna(ula_merged['pricing_scalar_current'])
ula_merged['pricing_scalar'] = ula_merged['pricing_scalar'].fillna(1.0)

print(f'ULA loans matched to groups: {len(ula_merged):,}')
print(f'Loans with non-default scalar: {(ula_merged.pricing_scalar != 1.0).sum():,}')

# Weighted average: sum(scalar * amt_financed) / sum(amt_financed) per group per quarter
def weighted_avg(g):
    return (g['pricing_scalar'] * g['amt_financed']).sum() / g['amt_financed'].sum()

wtd_vintage = ula_merged.groupby(['corporate_group', 'book_vintage']).apply(weighted_avg, include_groups=False)
wtd_vintage = wtd_vintage.unstack('book_vintage')

# Also compute weighted 'current' using latest available vintage's loans with current scalar
ula_latest = ula_slim.copy()
ula_latest = ula_latest.merge(dla_curr_fallback, on='dealer_number', how='left')
ula_latest['pricing_scalar_current'] = ula_latest['pricing_scalar_current'].fillna(1.0)
wtd_current = ula_latest.groupby('corporate_group').apply(
    lambda g: (g['pricing_scalar_current'] * g['amt_financed']).sum() / g['amt_financed'].sum(),
    include_groups=False,
)
wtd_vintage['current'] = wtd_current

# Add dealer count
wtd_vintage.insert(0, 'Dealers Matched', vintage_avg['Dealers Matched'])
wtd_vintage = wtd_vintage.rename_axis('Corporate Dealer Group')
wtd_vintage = wtd_vintage.sort_index()

print('\n=== WEIGHTED Average DLA Loss Scalar (by Amount Financed) ===')
display(wtd_vintage)

# --- Map weighted scalars to nearest dealer bucket ---
scalar_cols_w = [c for c in wtd_vintage.columns if c != 'Dealers Matched']
wtd_bucket_df = wtd_vintage[scalar_cols_w].map(scalar_to_bucket)
wtd_bucket_df.insert(0, 'Dealers Matched', wtd_vintage['Dealers Matched'])

print('\n=== WEIGHTED Scalar Mapped to Nearest Dealer Bucket ===')
display(wtd_bucket_df)

ULA loans matched to groups: 120,353
Loans with non-default scalar: 77,672

=== WEIGHTED Average DLA Loss Scalar (by Amount Financed) ===


book_vintage,Dealers Matched,2026 Q1,2026 Q2,2026 Q3,current
Corporate Dealer Group,,,,,
Auto Boutique,3,0.7588,0.7427,0.7302,0.7420
Avis,20,1.0000,1.0079,1.0012,1.0016
HGreg,10,0.9129,0.9542,0.9720,0.9574
Hertz Car Sales,111,0.9057,0.8785,0.8774,0.8826
Penske,149,0.9511,0.9445,0.9389,0.9443
Sonic Automotive,147,0.9355,0.9494,0.9233,0.9386
Woodhouse Auto Family,24,0.7338,0.7420,0.7195,0.7336



=== WEIGHTED Scalar Mapped to Nearest Dealer Bucket ===


book_vintage,Dealers Matched,2026 Q1,2026 Q2,2026 Q3,current
Corporate Dealer Group,,,,,
Auto Boutique,3,A,A,A,A
Avis,20,C,C,C,C
HGreg,10,B,C,C,C
Hertz Car Sales,111,B,B,B,B
Penske,149,C,C,C,C
Sonic Automotive,147,C,C,B,C
Woodhouse Auto Family,24,A,A,A,A


In [13]:
# =============================================================================
# CELL 8: EXPORT TO EXCEL
# =============================================================================

EXCEL_FILE = '../output/dealer_group_dla_scalars.xlsx'

with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl') as writer:
    output_df.to_excel(writer, sheet_name='Dealer Detail', index=False)
    vintage_avg.to_excel(writer, sheet_name='Group Summary')

    # Auto-fit column widths
    for sheet_name in writer.sheets:
        ws = writer.sheets[sheet_name]
        for col_cells in ws.columns:
            max_len = max(len(str(cell.value or '')) for cell in col_cells) + 2
            ws.column_dimensions[col_cells[0].column_letter].width = max_len

print(f'Exported to {EXCEL_FILE}')
print(f'  Sheet "Dealer Detail": {len(output_df)} rows (every dealer with its scalar)')
print(f'  Sheet "Group Summary": {len(vintage_avg)} rows (average scalar per group)')

CSV_FILE = '../output/dealer_group_dla_scalars.csv'
output_df.to_csv(CSV_FILE, index=False)
print(f'Exported to {CSV_FILE}')

Exported to ../output/dealer_group_dla_scalars.xlsx
  Sheet "Dealer Detail": 472 rows (every dealer with its scalar)
  Sheet "Group Summary": 8 rows (average scalar per group)
Exported to ../output/dealer_group_dla_scalars.csv
